In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.papm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'MIN': ['Anthony Edwards']}

Out Players:
{'DEN': ['Aaron Gordon', 'Peyton Watson'], 'MIN': ['Jaylen Clark', 'Terrence Shannon']}
Note: MIN (Timberwolves) has 4 confirmed players - lineup will still be updated
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 2 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age,STARTING
26440,NaN,3,2025-26,1631248,Baylor Scheierman,Baylor,1610612738,BOS,Boston Celtics,22501186,2026-04-12,BOS vs. ORL,W,38.806667,8,20,0.400,6,14,0.429,8,8,1.000,0,7,7,7,3,2,1,0,3,7,30,15,54.9,0,0,56.0,1,38:48,1,110.1,111.6,111.6,91.0,96.4,96.4,19.1,15.2,15.2,0.280,2.33,20.6,0.000,0.140,0.079,8.8,8.9,0.550,0.638,0.281,0.279,108.95,105.14,87.61,105.14,0.197,86,8.0,20.0,36,87,0.414,19,50,0.380,22,22,1.000,8,38,46,24,17.0,10,6,5,23,19,113,5.0,106.9,107.6,98.9,102.9,8.0,4.8,0.667,1.41,17.3,0.216,0.650,0.450,0.162,0.523,0.584,107.4,105.0,87.50,105,0.550,1610612753,ORL,Orlando Magic,36,91,0.396,12,43,0.279,24,30,0.800,14,36,50,22,19.0,6,5,6,19,23,108,-5.0,98.9,102.9,106.9,107.6,-8.0,-4.8,0.611,1.16,14.9,0.350,0.784,0.550,0.181,0.462,0.518,107.4,105.0,87.50,105,0.450,F,SG,25.0,1
26439,NaN,2,2025-26,1642382,Branden Carlson,Branden,1610612760,OKC,Oklahoma City Thunder,22501196,2026-04-12,OKC vs. PHX,L,42.233333,10,20,0.500,5,10,0.500,1,2,0.500,1,9,10,1,1,2,5,4,0,2,26,-18,59.5,1,0,56.0,1,42:14,1,111.3,112.5,112.5,131.2,134.5,134.5,-19.9,-22.0,-22.0,0.034,1.00,4.3,0.021,0.220,0.114,4.3,4.4,0.625,0.623,0.222,0.228,101.24,99.45,82.87,99.45,0.124,88,10.0,20.0,41,95,0.432,18,46,0.391,3,9,0.333,9,27,36,32,10.0,9,7,8,8,9,103,-32.0,103.0,105.1,130.3,137.8,-27.2,-32.7,0.780,3.20,22.4,0.228,0.587,0.388,0.102,0.526,0.520,101.8,98.0,81.67,98,0.403,1610612756,PHX,Phoenix Suns,56,101,0.554,20,41,0.488,3,6,0.500,13,41,54,28,13.0,5,8,7,9,8,135,32.0,130.3,137.8,103.0,105.1,27.2,32.7,0.500,2.15,19.3,0.413,0.772,0.612,0.133,0.653,0.651,101.8,98.0,81.67,98,0.597,C,C,26.0,1
26438,NaN,1,2025-26,1642847,Jeremiah Fears,Jeremiah,1610612740,NOP,New Orleans Pelicans,22501195,2026-04-12,NOP @ MIN,L,41.383333,12,28,0.429,0,5,0.000,12,14,0.857,3,7,10,5,2,2,1,3,3,8,36,0,62.5,1,0,57.0,1,41:23,1,116.9,119.8,119.8,122.3,118.6,118.6,-5.4,1.2,1.2,0.172,2.50,12.2,0.052,0.152,0.096,4.9,4.9,0.429,0.527,0.305,0.308,111.60,111.93,93.27,111.93,0.147,96,12.0,28.0,45,107,0.421,5,21,0.238,31,38,0.816,20,32,52,20,9.0,9,6,13,28,27,126,-6.0,111.8,117.8,123.5,122.2,-11.7,-4.5,0.444,2.22,13.1,0.394,0.692,0.525,0.084,0.444,0.509,109.8,107.5,89.58,107,0.432,1610612750,MIN,Minnesota Timberwolves,45,89,0.506,10,37,0.270,32,43,0.744,12,35,47,32,11.0,5,13,6,27,28,132,6.0,123.5,122.2,111.8,117.8,11.7,4.5,0.711,2.91,20.6,0.308,0.606,0.475,0.102,0.562,0.612,109.8,107.5,89.58,108,0.568,G,PG,

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260423_152941.json


,home_team,away_team,commence_time,bookmakers
0,Atlanta Hawks,New York Knicks,2026-04-23 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Toronto Raptors,Cleveland Cavaliers,2026-04-24 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Minnesota Timberwolves,Denver Nuggets,2026-04-24 01:30:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Philadelphia 76ers,Boston Celtics,2026-04-24 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Houston Rockets,Los Angeles Lakers,2026-04-25 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
pts_ast_df = pd.read_csv('data/processed/training/S26_TRAINING_PAPM.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
lines_dfs_pts_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points_assists')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()
pts_ast_names = lines_dfs_pts_ast['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
lines_us_pts_ast = lines_us[(lines_us['CATEGORY'] == 'player_points_assists')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-23 15:28:58
US latest pull: 2026-04-23 15:29:41


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Betr DFS,player_points,Josh Hart,Over,11.5,-137,2026-04-23,2026-04-23T22:28:34Z,2026-04-23 15:28:58
1,Betr DFS,player_points,Josh Hart,Under,11.5,-137,2026-04-23,2026-04-23T22:28:34Z,2026-04-23 15:28:58
2,Betr DFS,player_points,Jalen Brunson,Over,26.5,-137,2026-04-23,2026-04-23T22:28:34Z,2026-04-23 15:28:58
3,Betr DFS,player_points,Jalen Brunson,Under,26.5,-137,2026-04-23,2026-04-23T22:28:34Z,2026-04-23 15:28:58
4,Betr DFS,player_points,Mikal Bridges,Over,12.5,-137,2026-04-23,2026-04-23T22:28:34Z,2026-04-23 15:28:58


### Load my models

In [8]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-02-04.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-02-11.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [9]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Josh Hart,PTS,23.99,31.11,37.42,0.1554,0.3311,0.6108,3.73,10.30,22.86,"[0.726294552790854, 0.3558718861209964, 0.3287..."
1,Jalen Brunson,PTS,27.93,35.84,40.07,0.4847,0.7159,1.0309,13.54,25.66,41.31,"[0.569620253164557, 0.7345867949278531, 0.8523..."
2,Mikal Bridges,PTS,20.27,30.18,38.56,0.2201,0.4370,0.7461,4.46,13.19,28.77,"[0.3684886382669867, 0.327434318234973, 0.3070..."
3,OG Anunoby,PTS,24.07,32.47,37.70,0.2318,0.4311,0.7346,5.58,14.00,27.70,"[0.7138994050838292, 0.7296784550274846, 0.477..."
4,Karl-Anthony Towns,PTS,24.37,30.90,36.04,0.3896,0.6207,0.9205,9.50,19.18,33.17,"[0.7381889763779528, 0.9833762584874736, 0.699..."
5,Onyeka Okongwu,PTS,22.12,28.87,35.99,0.1605,0.4164,0.6308,3.55,12.02,22.70,"[0.4316546762589928, 0.1469687691365584, 0.403..."
6,Jalen Johnson,PTS,26.39,34.16,39.70,0.3147,0.5777,0.8387,8.31,19.73,33.29,"[0.7148530579825257, 0.5779816513761468, 0.661..."
7,Nickeil Alexander-Walker,PTS,28.12,35.15,40.55,0.2339,0.4698,0.7452,6.58,16.51,30.21,"[0.5879470847623715, 1.1131221719457012, 0.725..."
8,CJ McCollum,PTS,24.27,31.25,36.10,0.3372,0.6114,0.9029,8.18,19.10,32.60,"[1.112484548825711, 0.2843302443133951, 0.8470..."
9,Dyson Daniels,PTS,23.84,30.49,36.94,0.1730,0.3789,0.6175,4.12,11.55,22.81,"[0.3894297635605007, 0.2753872633390705, 0.420..."


### Get Line Probabilities

In [10]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
36,Luke Kennard,AST,3.5,19.46,27.43,35.07,0.70,4.06,8.68,0.460,0.540
5,Dyson Daniels,AST,5.0,23.84,30.49,36.94,0.88,4.28,9.10,0.484,0.516
29,Mitchell Robinson,AST,0.5,16.18,20.01,26.01,0.31,2.21,5.48,0.535,0.465
90,Julian Champagnie,REB,5.0,21.57,28.06,34.42,1.31,5.03,10.86,0.600,0.400
89,Devin Vassell,REB,4.0,24.91,32.52,37.59,1.05,3.90,9.34,0.558,0.442
72,Nikola Vučević,REB,5.0,16.81,21.39,30.37,2.71,6.10,13.42,0.682,0.318
225,Quentin Grimes,PTS,7.0,18.99,25.54,33.07,3.82,11.08,23.80,0.769,0.231
30,Scottie Barnes,AST,6.0,24.98,32.23,36.85,1.01,4.90,10.20,0.587,0.413
234,Jalen Williams,PTS,18.5,19.70,26.28,32.69,6.87,15.13,28.03,0.385,0.615
133,OG Anunoby,PTS,15.5,24.07,32.47,37.70,5.58,14.00,27.70,0.467,0.533


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
183,Reed Sheppard,PTS,9.5,21.61,29.07,35.66,5.16,15.05,30.50,0.868,0.132,PTS,Underdog,Los Angeles Lakers,-9.2,206.8,115.5,20.0,99.22,22.0,-117.0,-102.0,0.539,0.505,14.0,12.0,6.50,4.5,2.5,-0.692,0.756,0.244,40.22,-51.68,0.8,0.8,0.87,0.49,25.25,5.71,0.20,0.06,12.25,4.0
92,Victor Wembanyama,REB,11.5,20.41,26.67,31.90,4.06,8.18,16.06,0.511,0.489,REB,Underdog,Portland Trail Blazers,-1.5,220.5,113.5,12.0,101.63,9.0,-112.0,105.0,0.528,0.488,13.5,15.0,4.20,2.0,3.5,-0.476,0.683,0.317,29.28,-35.02,0.8,0.8,0.67,0.47,28.58,6.28,0.35,0.06,7.33,3.0
199,Duncan Robinson,PTS,10.5,18.84,26.30,33.47,3.80,11.60,24.84,0.756,0.243,PTS,Underdog,Orlando Magic,-2.5,213.8,113.6,13.0,100.56,14.0,-112.0,-111.0,0.528,0.526,14.3,13.5,4.19,3.8,3.0,-0.907,0.818,0.182,54.84,-65.40,0.8,0.8,0.73,0.55,26.57,3.17,0.17,0.05,7.86,7.0
134,Karl-Anthony Towns,PTS,19.5,24.37,30.90,36.04,9.50,19.18,33.17,0.585,0.415,PTS,Underdog,Atlanta Hawks,-1.5,214.5,112.9,10.0,102.50,5.0,-107.0,-105.0,0.517,0.512,20.2,21.0,4.37,0.7,1.5,-0.160,0.564,0.436,9.11,-14.88,0.8,0.7,0.73,0.63,29.06,4.04,0.27,0.06,28.14,7.0
198,Donovan Clingan,PTS,9.5,18.92,25.22,31.80,3.37,10.05,20.72,0.645,0.354,PTS,Underdog,San Antonio Spurs,1.5,220.5,110.4,3.0,100.72,12.0,-120.0,-103.0,0.545,0.507,10.1,9.0,5.45,0.6,-0.5,-0.110,0.544,0.456,-0.27,-10.13,0.8,0.5,0.67,0.42,26.28,3.60,0.16,0.05,9.57,7.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
71,Sam Hauser,REB,3.5,17.18,22.88,29.31,0.80,3.26,7.42,0.509,0.491,REB,PrizePicks,Philadelphia 76ers,-7.5,215.5,114.4,17.0,100.40,15.0,-117.0,-104.0,0.539,0.510,3.8,3.0,1.55,0.3,-0.5,-0.194,0.577,0.423,7.02,-17.03,0.6,0.4,0.47,0.43,26.95,5.02,0.13,0.04,3.86,7.0
189,Victor Wembanyama,PTS,25.5,20.41,26.67,31.90,11.46,22.69,36.13,0.572,0.428,PTS,PrizePicks,Portland Trail Blazers,-1.5,220.5,113.5,12.0,101.63,9.0,-115.0,-110.0,0.535,0.524,29.5,30.0,9.58,4.0,4.5,-0.418,0.662,0.338,23.77,-35.47,0.8,0.6,0.60,0.45,28.58,6.28,0.35,0.06,23.33,3.0
134,Karl-Anthony Towns,PTS,19.5,24.37,30.90,36.04,9.50,19.18,33.17,0.585,0.415,PTS,PrizePicks,Atlanta Hawks,-1.5,214.5,112.9,10.0,102.50,5.0,-107.0,-105.0,0.517,0.512,20.2,21.0,4.37,0.7,1.5,-0.160,0.564,0.436,9.11,-14.88,0.8,0.7,0.73,0.63,29.06,4.04,0.27,0.06,28.14,7.0
20,LeBron James,AST,8.5,24.16,31.41,36.80,0.99,5.08,10.26,0.489,0.511,AST,PrizePicks,Houston Rockets,9.2,206.8,112.1,6.0,96.98,29.0,-107.0,-110.0,0.517,0.524,9.1,9.5,3.75,0.6,1.0,-0.160,0.564,0.436,9.11,-16.76,0.6,0.6,0.47,0.36,31.95,6.70,0.27,0.09,5.50,6.0
153,Jamal Murray,PTS,28.5,30.12,36.20,40.29,10.84,21.87,35.36,0.322,0.678,PTS,PrizePicks,Minnesota Timberwolves,-1.5,232.0,112.5,8.0,101.50,10.0,-105.0,-116.0,0.512,0.537,27.6,24.0,11.10,-0.9,-4.5,0.081,0.468,0.532,-8.63,-0.94,0.2,0.4,0.40,0.25,37.94,4.10,0.25,0.06,24.14,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
91,De'Aaron Fox,REB,3.5,25.29,32.96,38.64,0.76,3.81,8.00,0.589,0.411,REB,Betr DFS,Portland Trail Blazers,-1.5,220.5,113.5,12.0,101.63,9.0,116.0,-125.0,0.463,0.556,3.5,3.5,2.42,0.0,0.0,0.000,0.500,0.500,8.00,-10.00,0.6,0.5,0.53,0.57,28.89,5.88,0.23,0.05,5.60,5.0
131,Jalen Brunson,PTS,26.5,27.93,35.84,40.07,13.54,25.66,41.31,0.500,0.500,PTS,Betr DFS,Atlanta Hawks,-1.5,214.5,112.9,10.0,102.50,5.0,-104.0,-104.0,0.510,0.510,24.3,25.5,6.96,-2.2,-1.0,0.316,0.376,0.624,-26.25,22.40,0.4,0.4,0.53,0.47,35.86,4.40,0.29,0.05,27.57,7.0
141,Max Strus,PTS,8.5,16.98,23.19,29.78,3.06,9.92,20.95,0.609,0.391,PTS,Betr DFS,Toronto Raptors,-2.5,221.5,112.1,5.0,99.22,21.0,-109.0,-103.0,0.522,0.507,10.4,9.0,9.35,1.9,0.5,-0.203,0.580,0.420,11.21,-17.22,0.4,0.5,0.60,0.58,24.23,3.61,0.17,0.05,13.00,2.0
151,Cameron Johnson,PTS,12.5,22.77,29.72,35.49,4.65,13.07,25.03,0.669,0.331,PTS,Betr DFS,Minnesota Timberwolves,-1.5,232.0,112.5,8.0,101.50,10.0,-104.0,-111.0,0.510,0.526,14.0,15.5,4.94,1.5,3.0,-0.304,0.619,0.381,21.42,-27.58,0.8,0.6,0.73,0.68,30.71,5.78,0.15,0.02,2.50,2.0
171,Paul George,PTS,18.5,24.68,31.90,37.73,8.87,18.82,33.53,0.507,0.493,PTS,Betr DFS,Boston Celtics,7.5,215.5,111.7,4.0,95.58,30.0,103.0,-114.0,0.493,0.533,21.0,20.5,8.99,2.5,2.0,-0.278,0.609,0.391,23.63,-26.60,0.4,0.7,0.53,0.35,31.12,5.95,0.26,0.05,14.50,2.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
157,Ayo Dosunmu,PTS,11.5,22.22,28.52,34.32,5.45,14.25,25.74,0.801,0.199,PTS,DraftKings Pick6,Denver Nuggets,1.5,232.0,116.0,21.0,99.49,20.0,100.0,-118.0,0.500,0.541,18.3,18.5,3.50,6.8,7.0,-1.943,0.974,0.026,94.80,-95.20,1.0,1.0,0.93,0.63,32.71,3.72,0.21,0.04,15.67,3.0
147,Scottie Barnes,PTS,18.5,24.98,32.23,36.85,6.91,16.40,28.39,0.392,0.608,PTS,DraftKings Pick6,Cleveland Cavaliers,2.5,221.5,114.1,15.0,100.70,13.0,-105.0,100.0,0.512,0.500,15.6,14.5,6.57,-2.9,-4.0,0.441,0.330,0.670,-35.57,34.00,0.2,0.3,0.27,0.43,30.43,4.39,0.21,0.04,18.14,7.0
133,OG Anunoby,PTS,15.5,24.07,32.47,37.70,5.58,14.00,27.70,0.467,0.533,PTS,DraftKings Pick6,Atlanta Hawks,-1.5,214.5,112.9,10.0,102.50,5.0,-115.0,-102.0,0.535,0.505,15.8,15.0,8.95,0.3,-0.5,-0.034,0.514,0.486,-3.90,-3.75,0.6,0.5,0.60,0.56,33.77,7.48,0.17,0.05,16.57,7.0
58,Jarrett Allen,REB,8.5,21.53,27.95,33.67,3.69,8.69,15.21,0.585,0.415,REB,DraftKings Pick6,Toronto Raptors,-2.5,221.5,112.1,5.0,99.22,21.0,114.0,-127.0,0.467,0.559,8.3,9.0,3.13,-0.2,0.5,0.064,0.474,0.526,1.44,-5.98,0.6,0.6,0.73,0.60,25.96,5.49,0.23,0.04,10.60,5.0
67,Jaden McDaniels,REB,4.5,24.47,32.69,37.83,1.00,4.13,8.77,0.439,0.561,REB,DraftKings Pick6,Denver Nuggets,1.5,232.0,116.0,21.0,99.49,20.0,110.0,-132.0,0.476,0.569,3.6,2.0,2.55,-0.9,-2.5,0.353,0.362,0.638,-23.98,12.13,0.6,0.4,0.33,0.55,30.42,7.83,0.21,0.08,4.25,8.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
53,Onyeka Okongwu,REB,7.5,22.12,28.87,35.99,2.67,7.22,14.99,0.399,0.601,REB,Underdog,New York Knicks,1.5,214.5,112.3,7.0,97.71,25.0,-105.0,-106.0,0.512,0.515,6.2,6.0,2.57,-1.3,-1.5,0.506,0.306,0.694,-40.26,34.87,0.2,0.3,0.40,0.55,29.75,6.01,0.17,0.05,9.25,8.0
176,Rui Hachimura,PTS,13.5,22.30,30.45,37.31,4.81,13.31,26.19,0.580,0.420,PTS,Underdog,Houston Rockets,9.2,206.8,112.1,6.0,96.98,29.0,-108.0,-102.0,0.519,0.505,12.5,13.5,6.47,-1.0,0.0,0.155,0.438,0.562,-15.64,11.30,0.6,0.5,0.40,0.39,25.80,7.00,0.17,0.05,8.33,6.0
144,Donovan Mitchell,PTS,27.5,28.46,34.69,38.82,13.79,25.61,40.10,0.513,0.486,PTS,Betr DFS,Toronto Raptors,-2.5,221.5,112.1,5.0,99.22,21.0,-102.0,-113.0,0.505,0.531,26.0,27.5,11.55,-1.5,0.0,0.130,0.448,0.552,-11.28,4.05,0.6,0.5,0.40,0.45,33.33,3.87,0.29,0.07,23.20,5.0
74,Tyrese Maxey,REB,3.5,29.01,36.42,40.32,0.49,3.46,8.02,0.482,0.518,REB,Underdog,Boston Celtics,7.5,215.5,111.7,4.0,95.58,30.0,-104.0,-111.0,0.510,0.526,4.1,3.0,2.60,0.6,-0.5,-0.231,0.591,0.409,15.93,-22.25,0.2,0.4,0.47,0.48,36.52,5.10,0.26,0.04,4.29,7.0
76,VJ Edgecombe,REB,5.5,29.32,36.03,40.21,1.39,5.12,10.54,0.620,0.380,REB,Betr DFS,Boston Celtics,7.5,215.5,111.7,4.0,95.58,30.0,-114.0,105.0,0.533,0.488,5.3,6.0,2.58,-0.2,0.5,0.078,0.469,0.531,-11.96,8.86,1.0,0.6,0.67,0.55,34.69,4.41,0.18,0.05,5.00,4.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 180  |  Pairs: 1063  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 85  |  Pairs: 418  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 64  |  Pairs: 143  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 153  |  Pairs: 484  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 180  |  Triples: 38368  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 85  |  Triples: 7600  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 153  |  Triples: 12722  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 64  |  Triples: 1389  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
